# 正则化详解

对应 Keras 版本：`Keras应用/第五章-机器学习基础/正则化详解.ipynb`

把正则化翻译成你能讲给别人听的版本：
- 为什么模型会过拟合
- 常见正则化方法到底在干什么
- 在 PyTorch 里怎么写
- 实验对比效果

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")

---
# 一、正则化到底想解决什么问题？

先记一句最重要的话：

> **正则化，就是想办法让模型别把训练集背得太死。**

深度学习模型能力很强，参数很多。如果训练数据不够多，或模型太大，就容易出现：
- 训练集上效果越来越好
- 验证集/测试集上效果却变差

这就叫**过拟合（Overfitting）**。

大白话：模型不是在学规律，而是在记题库答案，训练集一换，它就不会了。

所以正则化的目标就是：**让模型更克制一点，别乱记细节，尽量去学更通用的规律。**

---
# 二、把模型想成学生

| 概念 | 对应学生世界 |
|---|---|
| 训练集 | 平时练习题 |
| 验证集/测试集 | 考场新题 |
| 过拟合 | 死记硬背答案 |
| 正则化 | 不让他死记硬背，逼他理解题型 |

> 正则化不是让模型更猛，而是让模型更稳。

---
# 三、三种常见正则化方法

1. **减小网络规模** — 参数少一点，不容易死记硬背
2. **权重正则化（L1 / L2）** — 给大权重加惩罚，让模型别太激进
3. **Dropout** — 随机关掉神经元，不准只靠少数神经元

统一理解成一句话：

> **都是在给模型上约束，让它不要太自由。**

---
# 四、准备实验数据

用合成数据演示过拟合和正则化效果。

In [ ]:
# 生成过拟合友好的二分类数据（样本少、维度高）
torch.manual_seed(42)
np.random.seed(42)

N, D = 500, 20  # 样本少，维度适中
X = torch.randn(N, D)
true_w = torch.randn(D, 1)
logits = X @ true_w + 0.5 * torch.randn(N, 1)
y = (logits > 0).float().squeeze()

# 划分训练/验证
val_n = 100
train_X, val_X = X[val_n:], X[:val_n]
train_y, val_y = y[val_n:], y[:val_n]

print(f"训练集: {len(train_X)} 样本")
print(f"验证集: {len(val_X)} 样本")

---
# 五、方法一：减小网络规模

把模型做小一点——层数少一点，每层神经元少一点。

> 大白话：模型原来脑子太大，什么都硬记；现在脑容量限制一下，它只能记重点。

In [ ]:
# 偏大的模型（容易过拟合）
big_model = nn.Sequential(
    nn.Linear(20, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

# 更小的模型（更抗过拟合）
small_model = nn.Sequential(
    nn.Linear(20, 4),
    nn.ReLU(),
    nn.Linear(4, 4),
    nn.ReLU(),
    nn.Linear(4, 1)
)

print(f"大模型参数: {sum(p.numel() for p in big_model.parameters())}")
print(f"小模型参数: {sum(p.numel() for p in small_model.parameters())}")

In [ ]:
# 通用训练函数
def train_and_compare(model, train_X, train_y, val_X, val_y, epochs=200, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    
    train_losses, val_losses = [], []
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(train_X).squeeze(-1)
        loss = loss_fn(out, train_y)
        loss.backward()
        optimizer.step()
        
        train_losses.append(loss.item())
        
        with torch.no_grad():
            model.eval()
            val_out = model(val_X).squeeze(-1)
            val_loss = loss_fn(val_out, val_y).item()
            val_losses.append(val_loss)
    
    return train_losses, val_losses


# 训练大模型和小模型
big_train, big_val = train_and_compare(big_model, train_X, train_y, val_X, val_y)

# 重新初始化小模型
small_model = nn.Sequential(
    nn.Linear(20, 4), nn.ReLU(),
    nn.Linear(4, 4), nn.ReLU(),
    nn.Linear(4, 1)
)
small_train, small_val = train_and_compare(small_model, train_X, train_y, val_X, val_y)

# 绘制对比
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(big_train, label="big train")
plt.plot(big_val, label="big val")
plt.title("Big Model (64→64)")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(small_train, label="small train")
plt.plot(small_val, label="small val")
plt.title("Small Model (4→4)")
plt.xlabel("Epochs")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"大模型最终 - train: {big_train[-1]:.4f}, val: {big_val[-1]:.4f}")
print(f"小模型最终 - train: {small_train[-1]:.4f}, val: {small_val[-1]:.4f}")

---
# 六、方法二：权重正则化（L2 / weight_decay）

### Keras vs PyTorch 对比

| 框架 | L2 正则化写法 |
|------|-------------|
| Keras | `Dense(16, kernel_regularizer=regularizers.l2(0.001))` |
| PyTorch | `optim.Adam(model.parameters(), lr=0.01, weight_decay=0.001)` |

**L2 大白话**：如果权重太大，就罚你。逼模型不要过度依赖某几个特别大的权重。

> 类比：L2 就像老师说：别耍小聪明，给我稳一点、均衡一点。

### L1 vs L2

| | 惩罚什么 | 效果 |
|---|---|---|
| L1 | 权重绝对值之和 | 更容易把权重压成 0（特征选择） |
| L2 | 权重平方和 | 把权重整体压小（更平滑） |

In [ ]:
# L2 正则化：通过 weight_decay 参数

# 不使用 L2 正则化
model_no_reg = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1)
)

# 使用 L2 正则化（weight_decay=0.01）
model_l2 = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1)
)

# 训练对比
no_reg_train, no_reg_val = train_and_compare(model_no_reg, train_X, train_y, val_X, val_y, lr=0.01)

# 使用 weight_decay 训练 L2 模型
def train_with_weight_decay(model, train_X, train_y, val_X, val_y, weight_decay=0.01, epochs=200, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()
    train_losses, val_losses = [], []
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(train_X).squeeze(-1)
        loss = loss_fn(out, train_y)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        
        with torch.no_grad():
            model.eval()
            val_out = model(val_X).squeeze(-1)
            val_loss = loss_fn(val_out, val_y).item()
            val_losses.append(val_loss)
    
    return train_losses, val_losses

l2_train, l2_val = train_with_weight_decay(model_l2, train_X, train_y, val_X, val_y,
                                            weight_decay=0.01)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(no_reg_train, label="no reg train")
plt.plot(no_reg_val, label="no reg val")
plt.title("No Regularization")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(l2_train, label="L2 train")
plt.plot(l2_val, label="L2 val")
plt.title(f"L2 (weight_decay=0.01)")
plt.xlabel("Epochs")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"无正则化 - train: {no_reg_train[-1]:.4f}, val: {no_reg_val[-1]:.4f}")
print(f"L2正则化 - train: {l2_train[-1]:.4f}, val: {l2_val[-1]:.4f}")

In [ ]:
# L1 正则化：PyTorch 没有内置 weight_decay=L1，需要手动实现
# 方法：在每次 optimizer.step() 后手动减去 l1_penalty * sign(w)

model_l1 = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1)
)

def train_with_l1(model, train_X, train_y, val_X, val_y, l1_lambda=0.001, epochs=200, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    train_losses, val_losses = [], []
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(train_X).squeeze(-1)
        loss = loss_fn(out, train_y)
        
        # 手动添加 L1 惩罚项
        l1_penalty = torch.tensor(0.0)
        for param in model.parameters():
            l1_penalty = l1_penalty + torch.sum(torch.abs(param))
        loss = loss + l1_lambda * l1_penalty
        
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        
        with torch.no_grad():
            model.eval()
            val_out = model(val_X).squeeze(-1)
            val_loss = loss_fn(val_out, val_y).item()
            val_losses.append(val_loss)
    
    return train_losses, val_losses

l1_train, l1_val = train_with_l1(model_l1, train_X, train_y, val_X, val_y, l1_lambda=0.001)

print(f"L1正则化 - train: {l1_train[-1]:.4f}, val: {l1_val[-1]:.4f}")

---
# 七、方法三：Dropout

### Keras vs PyTorch 对比

| 框架 | Dropout 写法 |
|------|------------|
| Keras | `layers.Dropout(0.5)` |
| PyTorch | `nn.Dropout(0.5)` |

### 关键区别：`model.train()` vs `model.eval()`

在 PyTorch 中，Dropout 的行为取决于模型模式：
- `model.train()` → **开启** Dropout（训练时随机丢弃神经元）
- `model.eval()` → **关闭** Dropout（推理时所有神经元都工作）

这是与 Keras 最大的区别 —— Keras 自动处理，PyTorch 需要你手动切换。

In [ ]:
# Dropout 模型
dropout_model = nn.Sequential(
    nn.Linear(20, 64),
    nn.ReLU(),
    nn.Dropout(0.5),    # 训练时随机丢掉 50% 神经元
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(64, 1)
)

# Dropout 率选择：
# 0.2 丢得少，正则化轻
# 0.5 最常见
# 0.8 丢太多，学不动

dropout_train, dropout_val = train_and_compare(dropout_model, train_X, train_y, val_X, val_y)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(no_reg_train, label="no reg train")
plt.plot(no_reg_val, label="no reg val")
plt.title("No Regularization")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(dropout_train, label="dropout train")
plt.plot(dropout_val, label="dropout val")
plt.title("Dropout (0.5)")
plt.xlabel("Epochs")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"无正则化 - train: {no_reg_train[-1]:.4f}, val: {no_reg_val[-1]:.4f}")
print(f"Dropout   - train: {dropout_train[-1]:.4f}, val: {dropout_val[-1]:.4f}")

---
# 八、综合实验：组合多种正则化方法

In [ ]:
# 组合：小网络 + Dropout + L2
combined_model = nn.Sequential(
    nn.Linear(20, 32),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(32, 1)
)

combined_train, combined_val = train_with_weight_decay(
    combined_model, train_X, train_y, val_X, val_y,
    weight_decay=0.005, lr=0.005
)

print(f"{'方法':>15} | {'train loss':>10} | {'val loss':>10}")
print("-" * 40)
print(f"{'无正则化':>15} | {no_reg_train[-1]:>10.4f} | {no_reg_val[-1]:>10.4f}")
print(f"{'L2 (0.01)':>15} | {l2_train[-1]:>10.4f} | {l2_val[-1]:>10.4f}")
print(f"{'L1 (0.001)':>15} | {l1_train[-1]:>10.4f} | {l1_val[-1]:>10.4f}")
print(f"{'Dropout(0.5)':>15} | {dropout_train[-1]:>10.4f} | {dropout_val[-1]:>10.4f}")
print(f"{'组合':>15} | {combined_train[-1]:>10.4f} | {combined_val[-1]:>10.4f}")

---
# 九、三种方法对比总结

| 方法 | 它在干什么 | PyTorch 写法 | 大白话效果 |
|---|---|---|---|
| 减小网络规模 | 参数变少 | 减少 Linear 的 units | 不容易死记硬背 |
| L2正则化 | 给大权重加惩罚 | `weight_decay=0.001` | 让模型别太激进 |
| Dropout | 随机关掉神经元 | `nn.Dropout(0.5)` | 不准只靠少数神经元 |

### 容易混淆的问题

**为什么加了正则化后，训练误差反而更差了？**
这是正常的！正则化本来就是给模型增加限制。

**正则化是不是越强越好？**
不是。太强就变成欠拟合。类比做菜放盐：不放太淡，放一点刚好，放太多齁咸。

---
# 十、总逻辑

**一句话版本**：模型过拟合是因为对训练集学得太死；正则化通过各种方式限制模型，少记细节、多学规律，提升泛化能力。

**展开版本**：
1. 模型太强容易过拟合
2. 过拟合时训练集好但验证集差
3. 所以需要正则化
4. 正则化的本质是给模型加约束
5. 常见方法：减小网络规模、L1/L2、Dropout
6. 训练集没那么完美，但验证集更好

---

> **正则化的本质，就是牺牲一点训练集上的爽感，换来测试集上的靠谱。**

不让模型把答案背死，而是逼它学会做题的方法。